# 05. Paoay: harmonic features, classifier predictions and GMM clusters

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.pilots import climate, clustering, data, earth_engine, evaluation, features
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
subset_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
subset_run = Path(subset_run) if subset_run else tuning.latest_run(root, "subset", area=AREA)
rows, arrays = data.load(subset_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Subset:", subset_run, "| pixels:", len(rows))

In [ ]:
paoay = json.loads((subset_run / "paoay.json").read_text())
mask = arrays["paoay_mask"]
district_statistics_path = "data/paoay/benguet_vegetable_area_2024.csv"  # CSV with columns crop, area_ha; blank skips
p_cube = arrays["paoay_cube"].astype(float) * .0001
pixels = np.argwhere(mask)
p_values = p_cube[:, :, mask].transpose(2, 0, 1)
p_valid = np.isfinite(p_values).all(axis=2)
p_stack = features.stack(p_values)
print("Paoay year:", paoay["year"], "| study pixels:", len(pixels),
      "| mean clear half-months:", round(float(p_valid.sum(axis=1).mean()), 1))

## Coverage: harmonic features against the aligned green-up window

In [ ]:
paoay_sets = {"harmonic_full": features.harmonic_features(p_stack, p_valid),
              "harmonic_shift_invariant": features.harmonic_features(p_stack, p_valid, invariant=True)}
aligned_share = None
from experiments.transfer import transfer
latest = sorted((root / "pipeline_outputs/transfer").glob("paoay_check_*"), key=lambda p: p.stat().st_mtime)
if latest:
    status = pd.read_csv(latest[-1] / "status_shares.csv", index_col=0)
    aligned_share = 1 - status.loc[["No qualified green-up window", "Window not cloud-free"], "current"].sum()
display(pd.DataFrame([{"method": k, "pixels_with_features": int(ok.sum()), "share_of_study_pixels": round(float(ok.mean()), 3)}
                      for k, (_, ok, _) in paoay_sets.items()] +
                     [{"method": "aligned green-up window (transfer 12)", "pixels_with_features": None,
                       "share_of_study_pixels": None if aligned_share is None else round(float(aligned_share), 3)}]))

## Classifier trained on development parcels

In [ ]:
run = tuning.new_run(root, "paoay_gmm", subset_run / "meta.json", area=AREA)
rule = {"minimum_top_probability": .6, "minimum_margin": .1}
predictions = {}
for name, (pX, pok, _) in paoay_sets.items():
    invariant = name == "harmonic_shift_invariant"
    X, ok, _ = features.harmonic_features(cube, valid, invariant=invariant)
    train = np.asarray([r["partition"] in ("train", "validation") for r in rows]) & ok
    index = np.nonzero(train)[0]
    model = evaluation.fit(np.nan_to_num(X[index]), [rows[i] for i in index], {**evaluation.PARAMS, "n_estimators": 500})
    probabilities = model.predict_proba(np.nan_to_num(pX[pok]))
    ordered = np.sort(probabilities, axis=1)
    accepted = (ordered[:, -1] >= rule["minimum_top_probability"]) & (ordered[:, -1] - ordered[:, -2] >= rule["minimum_margin"])
    labels = np.where(accepted, model.classes_[np.argmax(probabilities, axis=1)], "Unknown crop type")
    predictions[name] = pd.Series("No harmonic fit", index=range(len(pixels)), dtype=object)
    predictions[name][np.nonzero(pok)[0]] = labels
shares = pd.DataFrame({k: v.value_counts(normalize=True) for k, v in predictions.items()}).fillna(0)
shares.to_csv(run / "classifier_shares.csv")
display((shares * 100).round(1))

## GMM clusters on shift-invariant harmonic features

In [ ]:
pX, pok, _ = paoay_sets["harmonic_shift_invariant"]
Z = np.nan_to_num(pX[pok])
model, scaler, bic = clustering.select_gmm(Z, components=range(2, 11))
mean_ari, min_ari = clustering.stability(Z, model.n_components, scaler)
clusters = pd.Series(-1, index=range(len(pixels)))
clusters[np.nonzero(pok)[0]] = model.predict(scaler.transform(Z))
tuning.save_records(run, "bic", bic)
print("Components chosen by BIC:", model.n_components, "| stability ARI mean", round(mean_ari, 3), "min", round(min_ari, 3))
display(pd.DataFrame(bic).round(0))

## Clusters against classifier predictions

In [ ]:
cross = pd.crosstab(clusters[pok_index := np.nonzero(pok)[0]].values,
                    predictions["harmonic_shift_invariant"][pok_index].values, rownames=["cluster"], colnames=["classifier"])
cross.to_csv(run / "clusters_vs_classifier.csv")
display(cross)

## Cluster profiles

In [ ]:
ndvi = data.indices(p_values)[..., 0]
profile = pd.DataFrame({"cluster": clusters.values, **{f"bin_{b:02d}": ndvi[:, b] for b in range(24)}})
medians = profile[profile.cluster >= 0].groupby("cluster").median()
medians.to_csv(run / "cluster_ndvi_profiles.csv")
figure, axis = plt.subplots(figsize=(10, 4))
for cluster, values in medians.iterrows():
    axis.plot(range(24), values.values, label=f"cluster {cluster}")
axis.set_xticks(range(0, 24, 2), [m for m in "JFMAMJJASOND"])
axis.set_ylabel("Median NDVI")
axis.legend(fontsize=8, ncol=2)
show_and_save(figure, run, "cluster_profiles")

## Clusters within reference areas

In [ ]:
from crop_pipeline.paoay import aligned
import rasterio
from rasterio.features import geometry_mask
from shapely.geometry import mapping
grid = paoay["grid"]
_, areas, _ = aligned.study_area([Path(p) for p in paoay["kmz_paths"]], json.loads((root / "settings.local.json").read_text())
                                 .get("paoay_roi_name", "Region of Interest"),
                                 list(json.loads((root / "settings.local.json").read_text()).get("paoay_context_names", [])), grid["crs"])
cluster_raster = np.full(mask.shape, -2)
cluster_raster[tuple(pixels.T)] = clusters.values
rows_out = []
for area in areas:
    inside = ~geometry_mask([mapping(area["geometry"])], out_shape=mask.shape, transform=rasterio.Affine(*grid["transform"])) & mask
    rows_out.append({"area": area["name"], "pixels": int(inside.sum()),
                     **{f"cluster_{c}": int((cluster_raster[inside] == c).sum()) for c in sorted(set(clusters))}})
display(pd.DataFrame(rows_out))

## District crop statistics

In [ ]:
if district_statistics_path and (root / district_statistics_path).is_file():
    statistics = pd.read_csv(root / district_statistics_path)
    display(statistics)
    statistics["share"] = statistics.area_ha / statistics.area_ha.sum()
    mapped = predictions["harmonic_shift_invariant"][predictions["harmonic_shift_invariant"].isin(evaluation.CROPS)]
    comparison = statistics.set_index("crop")[["share"]].join(mapped.value_counts(normalize=True).rename("map_share"))
    comparison.to_csv(run / "statistics_comparison.csv")
    display(comparison.round(3))
    cluster_crop = pd.crosstab(clusters[np.nonzero(pok)[0]].values, predictions["harmonic_shift_invariant"][np.nonzero(pok)[0]].values)
    crop_columns = [c for c in evaluation.CROPS if c in cluster_crop]
    if crop_columns:
        majority = cluster_crop[crop_columns].idxmax(axis=1)
        cluster_sizes = clusters[clusters >= 0].value_counts()
        by_cluster = pd.DataFrame({"pixels": cluster_sizes, "majority_crop": majority}).groupby("majority_crop").pixels.sum()
        display((by_cluster / by_cluster.sum()).rename("cluster_area_share").to_frame().join(statistics.set_index("crop")["share"]).round(3))
else:
    print("No district statistics file given; cluster-to-crop mapping by statistics was not run.")

## Maps

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 6))
image = np.full(mask.shape, np.nan)
image[tuple(pixels.T)] = np.where(clusters.values >= 0, clusters.values, np.nan)
axes[0].imshow(image, cmap="tab10", interpolation="nearest")
axes[0].set_title("GMM clusters")
codes = {"carrot": 1, "cabbage": 2, "potato": 3, "Unknown crop type": 4, "No harmonic fit": 5}
labels = np.full(mask.shape, np.nan)
labels[tuple(pixels.T)] = predictions["harmonic_shift_invariant"].map(codes).values
axes[1].imshow(labels, cmap="Set2", vmin=1, vmax=8, interpolation="nearest")
axes[1].set_title("Classifier (shift-invariant harmonics)")
for axis in axes:
    axis.axis("off")
show_and_save(figure, run, "paoay_maps")
print("Run folder:", run)